# Clase 4 · El Mundial de Qatar

**Taller en Ciencia de Datos y Aprendizaje Automático** · Especialización en Estadística · UNAL Manizales · 2026-II
Profesor: Carlos Manuel Orrego Franco

Este cuaderno lo resuelvo yo en vivo durante la primera mitad de la clase. Ustedes lo siguen, lo corren y me interrumpen. En la segunda mitad cada uno resuelve el gemelo, *Las calles de Bogotá*, con la misma técnica y otros datos.

| Ficha | |
|---|---|
| **Objetivo pedagógico** | Modelar un evento raro y entregar probabilidades en las que se pueda confiar: medir con reglas de puntuación en vez de exactitud, revisar la calibración y ver qué le pasa a la probabilidad cuando se «balancean» las clases. |
| **Enfoque** | Machine Learning. |
| **Resultados de aprendizaje** | 1 (formular el problema, los objetivos y la métrica), 2 (seleccionar y preparar datos), 4 (implementar y evaluar un modelo) y 5 (comunicar a quien no es técnico). |
| **Uso de IA** | Pueden pedirle a Claude que explique una línea de código, que derive una fórmula de geometría o que escriba una gráfica. Deben poder explicar sin ayuda por qué la exactitud no sirve con un evento raro, qué mide el puntaje de Brier, cómo se lee una curva de calibración, qué le hace el balanceo a las probabilidades y por qué una suma de probabilidades es un número esperado de goles. |
| **Entregable** | Este cuaderno corrido de principio a fin en su portafolio, con las predicciones escritas, la decisión en tres frases y la tabla de la sección 4.5 llenas. |

In [ ]:
# Celda de arranque. No hay que cambiarle nada: trae las herramientas del curso.
import pathlib, subprocess, sys
raiz = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "taller" / "datos.py").exists()), None)
if raiz is None:
    subprocess.run(["git", "clone", "-q", "https://github.com/cmorregof/taller-cd-ml-2026-2.git"], check=True)
    raiz = pathlib.Path("taller-cd-ml-2026-2")
sys.path.insert(0, str(raiz))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from taller import descargar, procedencia

plt.rcParams.update({"figure.figsize": (9, 4), "axes.spines.top": False, "axes.spines.right": False})
print("Herramientas listas.")

## 1. Formulación

En el fútbol casi ningún remate es gol. Por eso el marcador es un mal resumen de un partido: un equipo puede rematar veinte veces desde lejos y otro tres veces desde el punto penal. La pregunta que se hacen los analistas es cuánto **valía** cada remate, y la respuesta es una probabilidad. A eso se le llama gol esperado.

Imagínense la dirección deportiva de un club que usa el Mundial de Qatar como vitrina. Necesita tres respuestas:

1. **Predecir.** ¿Qué probabilidad de gol tiene cada remate, según dónde y cómo se pateó?
2. **Explicar.** ¿Qué hace peligroso un remate?
3. **Decidir.** ¿A qué delanteros manda a observar el club: a los que más goles hicieron o a los que más ocasiones de gol tuvieron?

**El dato.** La empresa StatsBomb publica el evento a evento de los 64 partidos del Mundial de 2022. De ahí sacamos todos los remates.

**La respuesta.** $y_i=1$ si el remate $i$ fue gol. El modelo entrega $\hat p_i$, la probabilidad estimada de gol.

**La métrica.** Aquí importa el número mismo, porque lo vamos a sumar. Si un jugador tuvo remates con probabilidades $\hat p_1,\dots,\hat p_m$, su número esperado de goles es

$$\widehat{\text{xG}}=\sum_{i=1}^{m}\hat p_i.$$

Esa suma solo significa algo si las probabilidades están **calibradas**: de los remates a los que el modelo les da 10 %, cerca del 10 % debe ser gol. Por eso medimos con dos reglas de puntuación, que castigan una probabilidad según qué tan lejos queda de lo que pasó:

$$\text{Brier}=\frac{1}{n}\sum_{i}(\hat p_i-y_i)^2,\qquad \text{pérdida logarítmica}=-\frac{1}{n}\sum_{i}\bigl[y_i\log\hat p_i+(1-y_i)\log(1-\hat p_i)\bigr].$$

En las dos, menos es mejor.

### Predicción escrita

Antes de ver un solo dato, escriban aquí su apuesta.

- De cada cien remates en un Mundial, ¿cuántos son gol? **Mi apuesta:**
- Un remate desde el borde del área grande, de frente al arco, ¿qué probabilidad de gol tiene? **Mi apuesta:**
- ¿Qué es más fácil de convertir desde la misma distancia, un remate con el pie o un cabezazo? **Mi apuesta:**

## 2. Datos

La descarga lee de GitHub los 64 partidos y tarda unos diez segundos. La licencia de StatsBomb pide citarlos como fuente y no permite redistribuir el dato, así que para esta clase no hay copia de respaldo: cada uno lo lee de GitHub.

In [ ]:
COMPETICION, TEMPORADA = 43, 106      # Mundial de Qatar 2022. Al final lo cambiarán por 223, 282: la Copa América de 2024

In [ ]:
remates = descargar("remates_mundial", competicion=COMPETICION, temporada=TEMPORADA)
print("Remates:", len(remates), "| Partidos:", remates["partido"].nunique(), "| Columnas:", remates.shape[1])
remates.head(3).T

### 2.1 Las trampas del dato

**Primera trampa: los penales de las tandas.** StatsBomb divide cada partido en periodos. Los periodos 1 y 2 son los dos tiempos, el 3 y el 4 son la prórroga, y el 5 es la tanda de penales.

In [ ]:
por_periodo = remates.groupby("periodo")["gol"].agg(remates="size", goles="sum", tasa="mean")
por_periodo["tasa"] = (por_periodo["tasa"] * 100).round(1)
print(por_periodo.to_string())

En la tanda, casi todos los remates son gol, porque todos son penales. Si se dejan, inflan la tasa de gol y le enseñan al modelo un remate que no existe durante el juego. **Decisión:** el periodo 5 sale.

**Segunda trampa: una columna que ya trae la respuesta de otro.** `statsbomb_xg` es el gol esperado que calcula la propia StatsBomb con su modelo. Si se la doy al mío, mi modelo aprende a copiar. **Decisión:** no entra como variable. La guardo para compararme con ella al final, que es para lo que sí sirve: es la referencia profesional.

**Tercera trampa: coordenadas que todavía no son variables.** Cada remate trae la posición $(x, y)$ en una cancha de 120 por 80 unidades, con el arco rival en $x=120$ y los palos en $y=36$ y $y=44$. Esas unidades son cercanas a una yarda. La posición cruda le dice poco a un modelo. Lo que importa es la geometría del remate: qué tan lejos está el arco y qué tan abierto se ve.

Sea $d_x=120-x$ y $d_y=|40-y|$. La distancia al centro del arco y el ángulo $\theta$ con que se ven los dos palos son

$$d=\sqrt{d_x^2+d_y^2},\qquad \theta=\arctan\!\left(\frac{8\,d_x}{d_x^2+d_y^2-16}\right),$$

donde el 8 es el ancho del arco y el 16 es el cuadrado de su mitad. Si el denominador es negativo, a $\theta$ se le suma $\pi$. Multiplico la distancia por 0,9144 para hablar en metros, sabiendo que es una aproximación.

In [ ]:
juego = remates[remates["periodo"] < 5].reset_index(drop=True)
dx, dy = 120 - juego["x"], (40 - juego["y"]).abs()

X = pd.DataFrame({
    "distancia_m": np.hypot(dx, dy) * 0.9144,
    "angulo": np.degrees(np.arctan2(8 * dx, dx**2 + dy**2 - 16)),
    "cabeza": (juego["parte_del_cuerpo"] == "Head").astype(int),
    "penal": (juego["tipo"] == "Penalty").astype(int),
    "tiro_libre": (juego["tipo"] == "Free Kick").astype(int),
    "bajo_presion": juego["bajo_presion"],
    "de_primera": juego["de_primera"],
    "mano_a_mano": juego["mano_a_mano"],
    "rivales_en_el_cono": juego["rivales_en_el_cono"],
    "arquero_en_el_cono": juego["arquero_en_el_cono"],
    "arquero_lejos_del_arco": juego["distancia_del_arquero_al_arco"],
})
y = juego["gol"]
variables = list(X.columns)
print(f"Remates en juego: {len(juego)} | goles: {y.sum()} | tasa de gol: {y.mean() * 100:.1f} %")
print("Faltantes por variable:"); print(X.isna().sum()[lambda s: s > 0].to_string())
print("Tipo de remate de las filas con faltantes:", juego.loc[X["rivales_en_el_cono"].isna(), "tipo"].value_counts().to_dict())

Comparen con su primera apuesta. Tres de las variables salen de la «foto» que StatsBomb toma en el instante del remate, con la posición de todos los jugadores: cuántos rivales hay dentro del triángulo que forman el balón y los dos palos, si el arquero está dentro de ese triángulo y qué tan lejos está de su arco. La descarga las calcula por ustedes.

Los faltantes tienen explicación: los penales no traen foto. No es un faltante al azar, es un faltante por diseño, y se llena con la mediana de entrenamiento dentro de cada modelo. La variable `penal` le avisa al modelo que esos remates son otra cosa.

### 2.2 Conocer el dato antes de modelarlo

Cuatro miradas: dónde se remata, cómo cae la tasa de gol con la distancia, cómo sube con el ángulo y qué tanto cambia según el tipo de remate. Con un evento raro hay que mirar siempre cuántos casos hay detrás de cada tasa. El error estándar de una tasa $\hat p$ estimada con $n$ remates es $\sqrt{\hat p(1-\hat p)/n}$.

In [ ]:
def tasa_por(grupo):
    tabla = pd.DataFrame({"n": y.groupby(grupo, observed=True).size(), "tasa": y.groupby(grupo, observed=True).mean() * 100})
    tabla["ee"] = np.sqrt(tabla["tasa"] * (100 - tabla["tasa"]) / tabla["n"])
    return tabla

fig, ejes = plt.subplots(2, 2, figsize=(13, 8))
eje = ejes[0, 0]
eje.scatter(juego.loc[y == 0, "x"], juego.loc[y == 0, "y"], s=8, color="#3E7CB1", alpha=0.35, label="no fue gol")
eje.scatter(juego.loc[y == 1, "x"], juego.loc[y == 1, "y"], s=22, color="#A8432C", label="gol")
eje.plot([120, 120], [36, 44], color="black", lw=4); eje.plot([102, 102, 120], [18, 62, 62], color="gray", lw=1); eje.plot([102, 120], [18, 18], color="gray", lw=1)
eje.set(xlim=(75, 121), ylim=(80, 0), title="Dónde se remató (el arco está a la derecha)", xlabel="x", ylabel="y"); eje.legend(fontsize=8)

por_distancia = tasa_por(pd.cut(X["distancia_m"], [0, 6, 9, 12, 16, 20, 25, 30, 60]))
ejes[0, 1].bar([str(i) for i in por_distancia.index], por_distancia["tasa"], yerr=2 * por_distancia["ee"], color="#16302B", ecolor="#E0A526")
ejes[0, 1].set(title="Tasa de gol según la distancia (m)", ylabel="%"); ejes[0, 1].tick_params(axis="x", rotation=30, labelsize=8)

por_angulo = tasa_por(pd.cut(X["angulo"], [0, 10, 15, 20, 30, 45, 180]))
ejes[1, 0].bar([str(i) for i in por_angulo.index], por_angulo["tasa"], yerr=2 * por_angulo["ee"], color="#16302B", ecolor="#E0A526")
ejes[1, 0].set(title="Tasa de gol según el ángulo con que se ve el arco (grados)", ylabel="%"); ejes[1, 0].tick_params(axis="x", rotation=30, labelsize=8)

clase = np.select([X["penal"] == 1, X["tiro_libre"] == 1, X["cabeza"] == 1, X["mano_a_mano"] == 1], ["penal", "tiro libre", "cabeza", "mano a mano"], "pie, en juego")
por_clase = tasa_por(pd.Series(clase, index=y.index)).sort_values("tasa")
ejes[1, 1].barh([f"{nombre} (n={int(n)})" for nombre, n in zip(por_clase.index, por_clase["n"])], por_clase["tasa"], xerr=2 * por_clase["ee"], color="#E0A526", ecolor="#16302B")
ejes[1, 1].set(title="Tasa de gol según la clase de remate", xlabel="%")
plt.tight_layout(); plt.show()
print(f"Mediana de la distancia: {X['distancia_m'].median():.1f} m | remates desde más de 20 m: {(X['distancia_m'] > 20).mean() * 100:.0f} %, con tasa de gol de {y[X['distancia_m'] > 20].mean() * 100:.1f} %")

La tasa de gol se desploma con la distancia y sube con el ángulo. Las barras de error son anchas donde hay pocos remates, que es cerca del arco y en los penales. Ese es el problema de fondo de un evento raro: hay muchos remates, pero pocos goles, y lo que se aprende depende de los pocos.

### 2.3 Agrupar sin etiquetas: ¿cuántas clases de remate hay?

Antes de mirar el gol, le pido a un algoritmo que agrupe los remates solo por su geometría. $K$-medias busca $K$ centros $\mu_1,\dots,\mu_K$ que minimicen la distancia de cada remate a su centro más cercano:

$$\min_{\mu_1,\dots,\mu_K}\ \sum_{i=1}^{n}\ \min_{k}\ \lVert z_i-\mu_k\rVert^2,$$

con $z_i$ la distancia y el ángulo estandarizados. El algoritmo no ve si el remate fue gol. Después miramos la tasa de gol de cada grupo.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

geometria = StandardScaler().fit_transform(X[["distancia_m", "angulo"]])
grupo = pd.Series(KMeans(n_clusters=4, n_init=10, random_state=0).fit_predict(geometria), index=X.index)
resumen_grupos = pd.DataFrame({"remates": grupo.value_counts(), "distancia media (m)": X.groupby(grupo)["distancia_m"].mean(),
                               "ángulo medio": X.groupby(grupo)["angulo"].mean(), "tasa de gol (%)": y.groupby(grupo).mean() * 100}).sort_values("distancia media (m)")

colores = dict(zip(resumen_grupos.index, ["#A8432C", "#E0A526", "#3E7CB1", "#16302B"]))
plt.figure(figsize=(7, 5))
plt.scatter(juego["x"], juego["y"], s=10, c=grupo.map(colores))
plt.plot([120, 120], [36, 44], color="black", lw=4); plt.xlim(75, 121); plt.ylim(80, 0); plt.title("Cuatro grupos de remates, armados sin mirar el gol"); plt.show()
display(resumen_grupos.round(1))

Los grupos salen ordenados por cercanía, y sus tasas de gol son muy distintas aunque el algoritmo nunca vio el gol. Eso dice dos cosas. La geometría sola ya separa remates buenos de malos, así que un modelo sencillo va a llegar lejos. Y la mayoría de los remates cae en los grupos donde casi nunca hay gol.

## 3. Modelado

### 3.1 La partición y la trampa de la exactitud

Los partidos se ordenan por fecha y se parten así:

$$\underbrace{\text{primer } 60\,\%\ \text{de los partidos}}_{\text{entrenamiento}}\ \Big|\ \underbrace{20\,\%\ \text{siguiente}}_{\text{validación: se escoge}}\ \Big|\ \underbrace{\text{último } 20\,\%}_{\text{prueba: se reporta}}$$

Se parte por partido y no por remate porque los remates de un mismo partido comparten rival, arquero y cancha. La prueba queda con las rondas finales.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score, brier_score_loss, log_loss

partidos = juego["partido"].drop_duplicates().tolist()          # ya vienen en orden de fecha
corte_1, corte_2 = int(0.6 * len(partidos)), int(0.8 * len(partidos))
E = juego["partido"].isin(partidos[:corte_1]).values
V = juego["partido"].isin(partidos[corte_1:corte_2]).values
P = juego["partido"].isin(partidos[corte_2:]).values
for nombre, tramo in [("Entrenamiento", E), ("Validación", V), ("Prueba", P)]:
    print(f"{nombre:14s} {juego['partido'][tramo].nunique():2d} partidos | {tramo.sum():4d} remates | {y[tramo].sum():3d} goles | tasa {y[tramo].mean() * 100:.1f} %")

In [ ]:
logistica = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(max_iter=2000)).fit(X[E], y[E])
prob = logistica.predict_proba(X[V])[:, 1]
real = y[V].values

def tres_medidas(real, dice):
    vp, fp, fn = ((dice == 1) & (real == 1)).sum(), ((dice == 1) & (real == 0)).sum(), ((dice == 0) & (real == 1)).sum()
    return {"exactitud": (dice == real).mean(), "sensibilidad": vp / (vp + fn), "precisión": vp / (vp + fp) if vp + fp else np.nan}

display(pd.DataFrame({"decir que ningún remate es gol": tres_medidas(real, np.zeros(len(real))),
                      "regresión logística, umbral 0,5": tres_medidas(real, prob >= 0.5)}).T.round(3))
print(f"Remates de validación: {len(real)} | goles: {real.sum()} | remates con probabilidad mayor que 0,5: {(prob >= 0.5).sum()}")

Decir que ningún remate es gol acierta casi nueve de cada diez veces y no sirve para nada. El modelo, puesto a decir sí o no con umbral de 0,5, casi nunca dice sí, porque casi ningún remate tiene más de 50 % de probabilidad. Esa es la marca de un problema **desbalanceado**: la clase que interesa es rara, y la exactitud la ignora.

La salida que uno encuentra primero en internet es «balancear las clases». La vamos a probar en la sección 3.4. Antes hay que cambiar la pregunta. Al club no le sirve que le digan gol o no gol. Le sirve la probabilidad, y por eso el modelo se mide con el puntaje de Brier y con la pérdida logarítmica de la sección 1.

### 3.2 La línea base

Dos referencias que no necesitan aprendizaje automático. La primera le da a todo remate la misma probabilidad, la tasa de gol del entrenamiento. La segunda es una regresión logística con una sola variable, la distancia.

In [ ]:
def puntuar(real, prob):
    prob = np.clip(np.asarray(prob, dtype=float), 1e-6, 1 - 1e-6)
    return {"pérdida logarítmica": log_loss(real, prob), "Brier": brier_score_loss(real, prob),
            "AUC": roc_auc_score(real, prob) if len(np.unique(prob)) > 1 else 0.5, "goles que predice": prob.sum(), "goles que hubo": int(np.sum(real))}

solo_distancia = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
referencias_validacion = {
    "tasa base": np.full(V.sum(), y[E].mean()),
    "solo distancia": solo_distancia.fit(X[["distancia_m"]][E], y[E]).predict_proba(X[["distancia_m"]][V])[:, 1],
}
probabilidades = {
    "tasa base": np.full(P.sum(), y[E | V].mean()),
    "solo distancia": solo_distancia.fit(X[["distancia_m"]][E | V], y[E | V]).predict_proba(X[["distancia_m"]][P])[:, 1],
}
errores = {nombre: {"validación": puntuar(y[V], referencias_validacion[nombre])["pérdida logarítmica"], **puntuar(y[P], probabilidades[nombre])} for nombre in probabilidades}
display(pd.DataFrame(errores).T.round(4))

La tasa base tiene un AUC de 0,5, porque no ordena nada, pero predice bien el **total** de goles. Esa propiedad vale mucho y es fácil perderla. Guárdenla para la sección 3.4.

### 3.3 Cinco modelos y un protocolo para escoger

| Modelo | Idea en una línea |
|---|---|
| Regresión logística | $\log\dfrac{p_i}{1-p_i}=\beta_0+x_i^\top\beta$ |
| $k$ vecinos | La fracción de goles entre los $k$ remates de entrenamiento más parecidos |
| Árbol de decisión | Preguntas de sí o no que van separando grupos con tasas de gol distintas |
| Bosque aleatorio | El promedio de muchos árboles entrenados sobre remuestreos del dato |
| *Gradient boosting* | Una suma de árboles pequeños, cada uno corrigiendo el error de los anteriores |

Gana quien tenga menor pérdida logarítmica en **validación**, y se reporta lo que haga en **prueba**. Uso 150 vecinos y hojas grandes a propósito. Con un evento raro, un vecindario pequeño o una hoja pequeña pueden no tener ningún gol, y el modelo diría probabilidad cero. Los remates lejanos entran cerca de 2 de cada 100 veces, así que un vecindario de 150 espera unos tres goles y uno de 40 espera menos de uno. Una probabilidad de cero que resulta ser gol recibe un castigo enorme en la pérdida logarítmica, y con razón, porque es afirmar que algo es imposible.

In [ ]:
def cinco_modelos(peso=None):
    llenar = lambda: SimpleImputer(strategy="median")
    return {
        "logística": make_pipeline(llenar(), StandardScaler(), LogisticRegression(max_iter=2000, class_weight=peso)),
        "k vecinos": make_pipeline(llenar(), StandardScaler(), KNeighborsClassifier(n_neighbors=150)),
        "árbol": make_pipeline(llenar(), DecisionTreeClassifier(max_depth=4, min_samples_leaf=20, class_weight=peso, random_state=0)),
        "bosque": make_pipeline(llenar(), RandomForestClassifier(n_estimators=400, min_samples_leaf=10, class_weight=peso, random_state=0, n_jobs=-1)),
        "boosting": make_pipeline(llenar(), HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=150, class_weight=peso, random_state=0)),
    }

for nombre, modelo in cinco_modelos().items():
    modelo.fit(X[E], y[E])
    en_validacion = puntuar(y[V], modelo.predict_proba(X[V])[:, 1])["pérdida logarítmica"]
    modelo.fit(X[E | V], y[E | V])
    probabilidades[nombre] = modelo.predict_proba(X[P])[:, 1]
    errores[nombre] = {"validación": en_validacion, **puntuar(y[P], probabilidades[nombre])}
todos = pd.DataFrame(errores).T

todos[["validación", "pérdida logarítmica"]].rename(columns={"pérdida logarítmica": "prueba"}).sort_values("validación").plot.barh(figsize=(9, 4), color=["#16302B", "#E0A526"])
plt.xlabel("pérdida logarítmica (menos es mejor)"); plt.title("Siete candidatos: validación para escoger, prueba para reportar"); plt.gca().invert_yaxis(); plt.show()
display(todos.sort_values("validación").round(4))

In [ ]:
mejor = todos["validación"].idxmin()
elegido = mejor if todos.loc[mejor, "pérdida logarítmica"] < todos.loc["solo distancia", "pérdida logarítmica"] else "solo distancia"
referencia = puntuar(y[P], juego["statsbomb_xg"][P])
print(f"Ganador en validación: {mejor}")
print(f"En prueba: pérdida logarítmica {todos.loc[mejor, 'pérdida logarítmica']:.4f} | Brier {todos.loc[mejor, 'Brier']:.4f} | predice {todos.loc[mejor, 'goles que predice']:.1f} goles y hubo {int(todos.loc[mejor, 'goles que hubo'])}")
print(f"Solo distancia: {todos.loc['solo distancia', 'pérdida logarítmica']:.4f} | tasa base: {todos.loc['tasa base', 'pérdida logarítmica']:.4f}")
print(f"El modelo de StatsBomb, en los mismos remates: pérdida logarítmica {referencia['pérdida logarítmica']:.4f} | Brier {referencia['Brier']:.4f} | predice {referencia['goles que predice']:.1f} goles")
print(f"Método con el que seguimos: {elegido}")

La regla es la de la clase 1: si el ganador de validación no le gana en la prueba a la línea base de la distancia, seguimos con la línea base, que es más simple.

Lean los tres renglones. La distancia sola ya recorre buena parte del camino desde la tasa base. En el Mundial de 2022 el ganador mejora un poco más. Y el modelo de StatsBomb, que es el producto de una empresa que vive de esto y que se entrenó con muchos más remates que los de un Mundial, queda por delante. La distancia entre el nuestro y el de ellos es lo que cuestan más datos y más variables.

Fíjense también en que las diferencias entre los cinco modelos en validación son pequeñas. Con unas decenas de goles en validación, un gol de más o de menos cambia el orden. Esa fragilidad es propia de los eventos raros.

### 3.4 La trampa de la clase: balancear rompe la probabilidad

Volvamos a la receta de internet. `class_weight="balanced"` le dice al modelo que cada gol pese tanto como todos los no goles juntos. Es lo mismo que repetir los goles hasta que haya tantos como no goles.

**Predicción escrita:** si entreno con las clases balanceadas, ¿cuántos goles predecirá el modelo en la prueba, donde hubo unas pocas decenas? **Mi apuesta:**

In [ ]:
def logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))

def armar(nombre, peso=None):
    """El modelo con ese nombre y las columnas que usa. La línea base de distancia es una logística de una sola variable."""
    if nombre == "solo distancia":
        return make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight=peso)), ["distancia_m"]
    return cinco_modelos(peso)[nombre], variables

(normal, columnas), (balanceado, _) = armar(elegido), armar(elegido, peso="balanced")
normal.fit(X[columnas][E], y[E]); balanceado.fit(X[columnas][E], y[E])
p_normal, p_balanceado = normal.predict_proba(X[columnas][P])[:, 1], balanceado.predict_proba(X[columnas][P])[:, 1]

# Recalibración de Platt: una regresión logística de una variable, ajustada en validación, que corrige la escala.
platt = LogisticRegression().fit(logit(balanceado.predict_proba(X[columnas][V])[:, 1]).reshape(-1, 1), y[V])
p_recalibrado = platt.predict_proba(logit(p_balanceado).reshape(-1, 1))[:, 1]

trampa = pd.DataFrame({"sin balancear": puntuar(y[P], p_normal), "balanceado": puntuar(y[P], p_balanceado),
                       "balanceado y recalibrado": puntuar(y[P], p_recalibrado)}).T
trampa["sensibilidad con umbral 0,5"] = [(p[y[P] == 1] >= 0.5).mean() for p in (p_normal, p_balanceado, p_recalibrado)]
print("Modelo:", elegido, "(entrenado solo con el tramo de entrenamiento, para dejar la validación libre)")
display(trampa.round(4))

Miren primero el AUC. Casi no cambia: el modelo balanceado **ordena** los remates igual de bien. Miren después los goles que predice. El balanceo le enseñó que la mitad de los remates son gol, y todas sus probabilidades quedaron infladas. La sensibilidad con umbral de 0,5 sube, y eso es lo que celebra quien recomienda la receta. Pero la suma ya no significa nada, y la pérdida logarítmica y el Brier lo castigan.

Hay una cuenta que lo explica. Si en el dato original la razón entre goles y no goles es $\pi/(1-\pi)$ y el balanceo la vuelve 1, las probabilidades quedan multiplicadas en la escala de la razón de probabilidades:

$$\frac{p_{\text{bal}}}{1-p_{\text{bal}}}\approx\frac{p}{1-p}\cdot\frac{1-\pi}{\pi}.$$

Con $\pi$ cercano a 0,11, ese factor es cercano a 8. La tercera fila deshace el daño con una **recalibración**: una regresión logística de una sola variable, ajustada con la validación, que devuelve las probabilidades a su escala. Funciona, pero es un rodeo para llegar adonde ya estaba el modelo sin balancear.

La regla que queda: balancear cambia el umbral, no la calidad del orden. Si lo que se necesita es una probabilidad, no se balancea. Si se necesita una decisión de sí o no, es más limpio dejar la probabilidad quieta y mover el umbral.

### 3.5 Calibración: ¿se le puede creer al número?

Agrupo los remates de la prueba en tramos según la probabilidad que les dio el modelo. En cada tramo $b$ comparo la probabilidad promedio $\bar p_b$ con la fracción real de goles $\bar y_b$. Un modelo calibrado cae sobre la diagonal. El resumen es el error de calibración esperado:

$$\text{ECE}=\sum_{b}\frac{n_b}{n}\,\bigl|\bar y_b-\bar p_b\bigr|.$$

In [ ]:
def calibracion(real, prob, tramos=5):
    cortes = pd.qcut(pd.Series(prob), tramos, duplicates="drop")
    tabla = pd.DataFrame({"prob": np.asarray(prob), "real": np.asarray(real)}).groupby(cortes.values, observed=True).agg(n=("real", "size"), predicha=("prob", "mean"), observada=("real", "mean"))
    return tabla, float((tabla["n"] / tabla["n"].sum() * (tabla["observada"] - tabla["predicha"]).abs()).sum())

curvas = {"sin balancear": p_normal, "balanceado": p_balanceado, "StatsBomb": juego["statsbomb_xg"][P].values}
plt.figure(figsize=(6.5, 5.5))
for (nombre, prob), color in zip(curvas.items(), ["#16302B", "#A8432C", "#3E7CB1"]):
    tabla, ece = calibracion(y[P], prob)
    plt.plot(tabla["predicha"], tabla["observada"], marker="o", color=color, label=f"{nombre} (ECE {ece:.3f})")
plt.plot([0, 1], [0, 1], color="gray", ls=":"); plt.xlim(0, 0.8); plt.ylim(0, 0.8)
plt.xlabel("probabilidad promedio que dio el modelo"); plt.ylabel("fracción de goles que hubo"); plt.title("Curva de calibración en la prueba"); plt.legend(); plt.show()
display(calibracion(y[P], p_normal)[0].round(3))

La curva del modelo balanceado queda muy por debajo de la diagonal: promete mucho más de lo que pasa. Las otras dos andan cerca. La tabla recuerda cuántos remates hay en cada tramo, y por eso los puntos de la derecha, que son los de pocos remates, bailan más.

### 3.6 Explicar: ¿qué hace peligroso un remate?

Con las variables estandarizadas, $e^{\beta_j}$ es la razón de probabilidades por cada desviación estándar de la variable $j$, con las demás quietas. La segunda gráfica muestra la probabilidad que el modelo elegido le da a un remate de pie, de frente al arco y en juego, según la distancia.

In [ ]:
explicativo = cinco_modelos()["logística"].fit(X[E | V], y[E | V])
razones = pd.Series(np.exp(explicativo[-1].coef_[0]), index=variables).sort_values()

modelo_final = armar(elegido)[0].fit(X[columnas][E | V], y[E | V])
distancias = np.arange(4, 36, 1.0)
de_frente = pd.DataFrame({"distancia_m": distancias, "angulo": np.degrees(np.arctan2(8 * distancias / 0.9144, (distancias / 0.9144) ** 2 - 16))})
for variable in variables[2:]:
    de_frente[variable] = X[variable][E | V].median()
de_frente["cabeza"] = 0; de_frente["penal"] = 0; de_frente["tiro_libre"] = 0

fig, (izq, der) = plt.subplots(1, 2, figsize=(13, 4.2))
razones.plot.barh(ax=izq, color=["#3E7CB1" if r < 1 else "#A8432C" for r in razones]); izq.axvline(1, color="gray")
izq.set(title="Razón de probabilidades por desviación estándar", xlabel="menor que 1 baja la probabilidad de gol; mayor que 1 la sube")
der.plot(distancias, modelo_final.predict_proba(de_frente[columnas])[:, 1] * 100, color="#A8432C", label="de pie")
der.plot(distancias, modelo_final.predict_proba(de_frente.assign(cabeza=1)[columnas])[:, 1] * 100, color="#3E7CB1", label="de cabeza")
der.axvline(16.5, color="gray", ls=":"); der.text(16.8, der.get_ylim()[1] * 0.9, "borde del área", fontsize=8, color="gray")
der.set(title=f"Probabilidad de gol de frente al arco ({elegido})", xlabel="distancia (m)", ylabel="%"); der.legend()
plt.tight_layout(); plt.show()
print(razones.round(2).to_string())

Comparen con sus apuestas segunda y tercera. Si el modelo elegido es de árboles, la curva de la derecha sale con escalones, porque un árbol predice por tramos. Dos advertencias para leer la gráfica de la izquierda.

La distancia y el ángulo van juntos: casi todo remate cercano tiene el arco abierto. La regresión reparte entre las dos un efecto que comparten, y ninguno de los dos coeficientes se puede leer solo.

Y lo que se ve es asociación. Que los remates bajo presión entren menos no dice que el jugador falle por la presión. Puede decir que solo se remata bajo presión cuando no había nada mejor.

## 4. Comunicación

### 4.1 Decidir: ¿a quién mandar a observar?

Para juzgar a un delantero no sirve el modelo que ya vio sus remates. Uso predicciones **fuera de muestra**: parto los partidos en ocho grupos, y la probabilidad de cada remate la calcula un modelo entrenado con los otros siete. Así cada remate recibe un número de un modelo que no lo conocía.

Con esas probabilidades, el gol esperado de un jugador es la suma de la sección 1. Para saber si sus goles se salen de lo esperable, simulo: cada remate entra con su probabilidad, diez mil veces, y miro entre qué valores cae el total el 90 % de las veces. Si los goles reales caen dentro de esa franja, la diferencia con el gol esperado se puede explicar por la suerte de un torneo corto.

In [ ]:
from sklearn.model_selection import GroupKFold

fuera_de_muestra = np.zeros(len(juego))
for entrena, prueba in GroupKFold(n_splits=8).split(X, y, juego["partido"]):
    fuera_de_muestra[prueba] = armar(elegido)[0].fit(X[columnas].iloc[entrena], y.iloc[entrena]).predict_proba(X[columnas].iloc[prueba])[:, 1]
juego["xg"] = fuera_de_muestra
total = puntuar(y, fuera_de_muestra)
print(f"Todo el torneo, fuera de muestra: predice {total['goles que predice']:.1f} goles y hubo {total['goles que hubo']} | pérdida logarítmica {total['pérdida logarítmica']:.4f} | StatsBomb: {puntuar(y, juego['statsbomb_xg'])['pérdida logarítmica']:.4f}")

sorteo = np.random.default_rng(0)
def franja(probabilidades_jugador):
    simulados = (sorteo.random((10000, len(probabilidades_jugador))) < probabilidades_jugador.values).sum(axis=1)
    return np.percentile(simulados, [5, 95])

jugadores = juego.groupby(["jugador", "equipo"]).agg(remates=("gol", "size"), goles=("gol", "sum"), xg=("xg", "sum"), xg_statsbomb=("statsbomb_xg", "sum")).reset_index()
jugadores = jugadores.sort_values("xg", ascending=False).head(12).reset_index(drop=True)
franjas = [franja(juego.loc[juego["jugador"] == nombre, "xg"]) for nombre in jugadores["jugador"]]
jugadores["franja de 90 %"] = [f"{int(a)} a {int(b)}" for a, b in franjas]
jugadores["dentro de la franja"] = [a <= g <= b for g, (a, b) in zip(jugadores["goles"], franjas)]
display(jugadores.round(2))
print("\nLos que más goles hicieron:")
print(juego.groupby("jugador").agg(goles=("gol", "sum"), xg=("xg", "sum"), remates=("gol", "size")).sort_values("goles", ascending=False).head(6).round(2).to_string())

La tabla de arriba ordena por ocasiones y la de abajo por goles. No coinciden del todo. El argumento para la dirección deportiva es que las ocasiones se repiten de un torneo a otro más que la puntería, y que un club paga por lo que se repite. Este cuaderno no puede comprobar esa afirmación, porque tiene un solo torneo. Va como supuesto en la decisión y como pregunta en los riesgos.

Fíjense en lo anchas que son las franjas. Con veinte o treinta remates, hacer tres goles o hacer siete puede ser el mismo jugador con distinta suerte.

### La decisión en tres frases

Escríbanla para un director deportivo que sabe de fútbol y no sabe qué es una pérdida logarítmica. Una frase para lo que recomiendan, una para la evidencia y una para lo que puede salir mal.

1. **Recomendación:**
2. **Evidencia:**
3. **Riesgo:**

### 4.2 Bitácora de decisiones

| Decisión | Alternativa descartada | Por qué |
|---|---|---|
| Fuera el periodo 5 | Dejar las tandas de penales | No son remates del juego e inflan la tasa de gol |
| `statsbomb_xg` no es variable de entrada | Usarla | Es la salida de otro modelo: sirve de referencia, no de insumo |
| Distancia y ángulo en vez de $x$ e $y$ | Coordenadas crudas | La geometría del remate es lo que importa |
| Partición por partido y por fecha | Por remate, al azar | Los remates de un partido comparten rival y arquero |
| Pérdida logarítmica para escoger | Exactitud | Con 11 % de goles la exactitud premia decir siempre que no |
| 150 vecinos y hojas grandes | Valores por defecto | Un vecindario sin goles da probabilidad cero |
| No balancear las clases | `class_weight="balanced"` | Infla las probabilidades y la suma deja de ser un número de goles |
| Predicciones fuera de muestra para juzgar jugadores | Las del modelo final | Un modelo no debe calificar los remates con que se entrenó |
| Franja por simulación | Comparar goles con xG a ojo | Con pocos remates la suerte pesa mucho |

### 4.3 Riesgos y mantenimiento

Respondan en dos o tres líneas cada una.

- El modelo se entrenó con unos mil remates y unos cien goles. ¿En qué clase de remate confían menos en su probabilidad? Miren la curva de calibración.
- El modelo no sabe quién patea. ¿A qué jugadores subestima de forma sistemática un gol esperado que trata igual a todos?
- Un Mundial son siete partidos por equipo, a lo sumo. ¿Qué habría que tener para comprobar que las ocasiones se repiten más que los goles?
- ¿Qué pasaría con sus conclusiones si StatsBomb cambiara la manera de anotar la posición del arquero?
- Si el club usara este número para fijar el salario de un jugador, ¿qué le exigirían al modelo que hoy no le exigen?

### 4.4 Procedencia

In [ ]:
print("Fuente: StatsBomb Open Data (github.com/statsbomb/open-data). Se cita a StatsBomb y no se redistribuye el dato.")
procedencia()["remates_mundial"]

### 4.5 Cambien una línea

Vuelvan a la celda donde dice `COMPETICION, TEMPORADA = 43, 106`, pongan `COMPETICION, TEMPORADA = 223, 282` y corran todo de nuevo con *Entorno de ejecución*, *Ejecutar todo*. Esa es la Copa América de 2024. Si prefieren, el Mundial femenino de 2023 es `72, 107`. Llenen la tabla con sus dos corridas.

| | Mundial 2022 | Su torneo |
|---|---|---|
| Partidos, remates en juego y goles | | |
| Tasa de gol | | |
| Ganador en validación | | |
| Pérdida logarítmica en prueba: ganador, solo distancia y StatsBomb | | |
| Goles que predice el modelo balanceado en la prueba, y goles que hubo | | |
| Jugador con más gol esperado, sus goles y su franja | | |

Con la tabla llena, respondan en tres líneas. ¿Qué resultados se repiten de un torneo al otro y cuáles cambian? ¿Qué le pasa al protocolo de validación cuando el torneo tiene la mitad de los partidos? **Mi respuesta:**